In [1]:
# 统一论文图的风格和尺寸, 方便后续拼接
import os
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm   # 修改 1：新增，用于手动加载 Arial.ttf


# =========================
# 修改 2：只需要改这里
# =========================
ARIAL_TTF_PATH = "/data/jychen/package/Arial/arial.ttf"   # 改成你的 arial.ttf 实际路径
USE_ARIAL_TTF = True                # True = 从 ttf 文件加载 Arial；False = 直接用系统字体名


_PUB_FONT_NAME = None


def load_pub_font():
    """
    手动加载 Arial.ttf，并返回字体在 matplotlib 中识别到的真实名称。
    这样可以避免系统装了 Arial 但 matplotlib 仍然找不到的问题。
    """
    global _PUB_FONT_NAME

    if _PUB_FONT_NAME is not None:
        return _PUB_FONT_NAME

    if USE_ARIAL_TTF:
        if not os.path.exists(ARIAL_TTF_PATH):
            raise FileNotFoundError(f"找不到 Arial 字体文件: {ARIAL_TTF_PATH}")

        fm.fontManager.addfont(ARIAL_TTF_PATH)
        _PUB_FONT_NAME = fm.FontProperties(fname=ARIAL_TTF_PATH).get_name()
    else:
        _PUB_FONT_NAME = "Arial"

    print(f"Using font: {_PUB_FONT_NAME}")
    return _PUB_FONT_NAME


def mm_to_inch(mm):
    return mm / 25.4


FIG_SIZE = {
    "single": 90,
    "double": 180,
}


FIG_PRESETS = {
    "single_small":  ("single", 60),
    "single_medium": ("single", 75),
    "double_short":  ("double", 70),
    "double_medium": ("double", 100),
    "double_large":  ("double", 140),
}


_SHARED_FONT_SIZES = {
    "font.size": 9,
    "axes.labelsize": 10,
    "axes.titlesize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 8,
}


def set_pub_style():
    pub_font = load_pub_font()   # 修改 3：每次设置风格前，先确保 Arial.ttf 已经加载

    base = {
        # 修改 4：不再直接写死 "Arial"，而是使用 ttf 文件真实识别到的字体名
        "font.family": pub_font,
        "font.sans-serif": [pub_font, "Arial", "Helvetica", "Liberation Sans", "DejaVu Sans"],

        "font.weight": "normal",
        "axes.labelweight": "normal",
        "axes.titleweight": "normal",

        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",

        # 稍粗一些的论文图风格
        "axes.linewidth": 1.5,
        "xtick.major.width": 1.4,
        "ytick.major.width": 1.4,
        "xtick.minor.width": 1.1,
        "ytick.minor.width": 1.1,
        "xtick.major.size": 4.5,
        "ytick.major.size": 4.5,
        "xtick.minor.size": 2.8,
        "ytick.minor.size": 2.8,
        "xtick.direction": "in",
        "ytick.direction": "in",

        "lines.linewidth": 1.7,
        "patch.linewidth": 1.3,

        "legend.frameon": False,
        "legend.handlelength": 1.5,
        "legend.handletextpad": 0.4,

        "axes.unicode_minus": False,
        "axes.spines.top": True,
        "axes.spines.right": True,

        "savefig.dpi": 600,
        "figure.dpi": 120,
    }

    base.update(_SHARED_FONT_SIZES)
    mpl.rcParams.update(base)


def new_figure(mode="single", height_mm=65):
    set_pub_style()
    width_mm = FIG_SIZE[mode]
    fig, ax = plt.subplots(
        figsize=(mm_to_inch(width_mm), mm_to_inch(height_mm))
    )
    return fig, ax


def new_subplots(nrows, ncols, preset="double_medium", **kwargs):
    set_pub_style()
    mode, height_mm = FIG_PRESETS[preset]
    width_mm = FIG_SIZE[mode]

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(mm_to_inch(width_mm), mm_to_inch(height_mm)),
        **kwargs
    )
    return fig, axes


def savefig_pub(fig, filename, dpi=600, tight=False):
    if tight:
        fig.savefig(filename, dpi=dpi, bbox_inches="tight", pad_inches=0.02)
    else:
        fig.savefig(filename, dpi=dpi)


# 应用统一风格模板
set_pub_style()

Using font: Arial


In [2]:
import MDAnalysis as mda
import numpy as np
import sys
from scipy.spatial import cKDTree
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from multiprocessing import Pool, cpu_count
import time

# ==========================================================
# 1. 核心工作函数 (单帧处理)
# ==========================================================

def process_frame_task(args):
    """
    处理单帧的并行函数
    """
    frame_idx, positions, atom_to_chain_map, cutoff, temp_K, kB = args
    
    n_atoms = len(positions)
    if n_atoms == 0:
        return None

    # --- A. 空间团簇识别 (KDTree) ---
    tree = cKDTree(positions)
    pairs = tree.query_pairs(r=cutoff, output_type='ndarray')
    
    if len(pairs) == 0:
        return None

    data = np.ones(len(pairs))
    row = pairs[:, 0]
    col = pairs[:, 1]
    adj_matrix = csr_matrix((data, (row, col)), shape=(n_atoms, n_atoms))
    
    n_components, labels = connected_components(adj_matrix, directed=False)
    counts = np.bincount(labels)
    largest_label = np.argmax(counts)
    cluster_indices = np.where(labels == largest_label)[0]
    
    # --- B. 统计链的数量 (实时记录) ---
    chains_in_cluster = np.unique(atom_to_chain_map[cluster_indices])
    n_chains_in_cluster = len(chains_in_cluster)
    
    # --- C. 计算几何张量 ---
    if len(cluster_indices) < 10:
        return None
        
    cluster_pos = positions[cluster_indices]
    N_cluster = len(cluster_indices)
    
    r_cms = np.mean(cluster_pos, axis=0)
    dr = cluster_pos - r_cms
    C = np.dot(dr.T, dr) / N_cluster
    eigvals = np.linalg.eigvalsh(C)
    
    return {
        "frame": frame_idx,
        "eigvals": eigvals,
        "n_atoms": N_cluster,
        "n_chains": n_chains_in_cluster # 返回当前帧的链数
    }

# ==========================================================
# 2. 主分析函数
# ==========================================================

def run_analysis(tpr_path, xtc_path, selection="protein", cutoff=8.0, 
                 temp_K=310.0, block_size_frames=2000, start=0, end=None, step=1, n_jobs=-1):
    
    kB = 1.380649e-23
    u = mda.Universe(tpr_path, xtc_path)
    prot = u.select_atoms(selection)
    dt = u.trajectory.dt
    
    print(f"开始分析... 温度: {temp_K} K")
    print(f"Selection: {selection}")
    print(f"Block Size: {block_size_frames}, Cluster Cutoff: {cutoff} A")

    # 1. 建立链映射 (Topology Recognition)
    fragments = prot.fragments
    n_total_chains = len(fragments)
    atom_to_chain_map = np.zeros(prot.n_atoms, dtype=int)
    for i, frag in enumerate(fragments):
        _, _, mask = np.intersect1d(frag.indices, prot.indices, return_indices=True)
        atom_to_chain_map[mask] = i

    # 2. 准备并行任务
    trajectory = u.trajectory[start:end:step]
    n_frames = len(trajectory)
    task_args = []
    for ts in trajectory:
        task_args.append((
            ts.frame, 
            prot.positions.copy(), 
            atom_to_chain_map,
            cutoff, 
            temp_K, 
            kB
        ))
    
    # 3. 执行并行计算
    if n_jobs == -1: n_jobs = cpu_count()
    print("正在收集轨迹几何数据...")
    
    results_raw = []
    with Pool(processes=n_jobs) as pool:
        for i, res in enumerate(pool.imap(process_frame_task, task_args), 1):
            results_raw.append(res)
            if i % 10 == 0 or i == n_frames:
                sys.stdout.write(f"\r处理进度: {i}/{n_frames} ({i/n_frames*100:.1f}%)")
                sys.stdout.flush()
    
    print() 

    # 4. 数据整理
    valid_results = [r for r in results_raw if r is not None]
    if not valid_results:
        print("错误：未识别到有效团簇")
        return

    avg_atoms = np.mean([r['n_atoms'] for r in valid_results])
    avg_chains_total = np.mean([r['n_chains'] for r in valid_results])
    
    print(f"平均团簇包含粒子数: {avg_atoms:.1f} (总粒子数: {len(prot)})")
    print(f"平均团簇包含链数量: {avg_chains_total:.1f} (系统总链数: {n_total_chains})")
    print("数据收集完成。开始分块计算表面张力并分析链数变化...")
    
    # 5. 分块计算
    eigvals_hist = np.array([r['eigvals'] for r in valid_results])
    frames_hist = np.array([r['frame'] for r in valid_results])
    chains_hist = np.array([r['n_chains'] for r in valid_results])
    
    for i in range(0, len(valid_results), block_size_frames):
        block_eig = eigvals_hist[i : i + block_size_frames]
        block_frames = frames_hist[i : i + block_size_frames]
        block_chains = chains_hist[i : i + block_size_frames]
        
        if len(block_eig) < 100:
            continue
            
        # 半径 R
        raw_axes = np.sqrt(5 * block_eig)
        R_inst = np.prod(raw_axes, axis=1)**(1.0/3.0)
        R_avg = np.mean(R_inst)
        
        # 链数统计
        c_avg = np.mean(block_chains)
        c_min = np.min(block_chains)
        c_max = np.max(block_chains)
        
        # 表面张力计算
        l1, l2, l3 = block_eig[:, 0], block_eig[:, 1], block_eig[:, 2]
        a = R_avg * (l1**(1/3)) / ((l2 * l3)**(1/6))
        b = R_avg * (l2**(1/3)) / ((l1 * l3)**(1/6))
        c = R_avg * (l3**(1/3)) / ((l1 * l2)**(1/6))
        da, db, dc = a - R_avg, b - R_avg, c - R_avg
        var_plus = (np.mean((da + db)**2) + np.mean((db + dc)**2) + np.mean((dc + da)**2)) / 3.0
        var_minus = (np.mean((da - db)**2) + np.mean((db - dc)**2) + np.mean((dc - da)**2)) / 3.0
        
        ang2_to_m2 = 1e-20
        gamma_20 = (15 * kB * temp_K) / (16 * np.pi * ang2_to_m2 * var_plus) * 1000
        gamma_22 = (45 * kB * temp_K) / (16 * np.pi * ang2_to_m2 * var_minus) * 1000
        gamma_avg = (gamma_20 + gamma_22) / 2.0
        
        # 格式化输出
        start_ns = block_frames[0] * dt / 1000
        end_ns = block_frames[-1] * dt / 1000
        
        print(f"Block [{start_ns:.1f}-{end_ns:.1f} ns]: "
              f"Chains={c_avg:.1f}[{c_min}-{c_max}], R={R_avg:.1f}Å, "
              f"γ={gamma_avg:.3f} mN/m (γ20={gamma_20:.3f}, γ22={gamma_22:.3f})")

# ==========================================================
# 3. 运行设置
# ==========================================================

if __name__ == "__main__":
    # 请确保路径正确
    TPR = "/data/jychen/MD_projects/curvature_phase/Alpha-Synuclein/martini3001/Martini_alhx/mdrun/M3IDP/W_cluster/alhx_0/TR_Mdvwhole/pbc.tpr"
    XTC = "/data/jychen/MD_projects/curvature_phase/Alpha-Synuclein/martini3001/Martini_alhx/mdrun/M3IDP/W_cluster/alhx_0/TR_Mdvwhole/clus_centered.xtc"
    
    run_analysis(
        tpr_path=TPR,
        xtc_path=XTC,
        selection="protein",
        cutoff=8.0,
        temp_K=310.0,
        block_size_frames=2000, 
        start=10000,
        step=1,
        n_jobs=-1 
    )

/home/jychen/anaconda3/envs/CJY/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


开始分析... 温度: 310.0 K
Selection: protein
Block Size: 2000, Cluster Cutoff: 8.0 A
正在收集轨迹几何数据...
处理进度: 10001/10001 (100.0%)
平均团簇包含粒子数: 14750.0 (总粒子数: 14750)
平均团簇包含链数量: 50.0 (系统总链数: 50)
数据收集完成。开始分块计算表面张力并分析链数变化...
Block [10000.0-11999.0 ns]: Chains=50.0[50-50], R=95.3Å, γ=0.981 mN/m (γ20=0.972, γ22=0.991)
Block [12000.0-13999.0 ns]: Chains=50.0[50-50], R=96.3Å, γ=0.293 mN/m (γ20=0.285, γ22=0.301)
Block [14000.0-15999.0 ns]: Chains=50.0[50-50], R=97.3Å, γ=0.454 mN/m (γ20=0.445, γ22=0.464)
Block [16000.0-17999.0 ns]: Chains=50.0[50-50], R=96.4Å, γ=0.683 mN/m (γ20=0.676, γ22=0.690)
Block [18000.0-19999.0 ns]: Chains=50.0[50-50], R=96.1Å, γ=1.067 mN/m (γ20=1.059, γ22=1.075)


In [3]:
import MDAnalysis as mda
import numpy as np
import sys
from scipy.spatial import cKDTree
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from multiprocessing import Pool, cpu_count
import time

# ==========================================================
# 1. 核心工作函数 (单帧处理)
# ==========================================================

def process_frame_task(args):
    """
    处理单帧的并行函数
    """
    frame_idx, positions, atom_to_chain_map, cutoff, temp_K, kB = args
    
    n_atoms = len(positions)
    if n_atoms == 0:
        return None

    # --- A. 空间团簇识别 (KDTree) ---
    tree = cKDTree(positions)
    pairs = tree.query_pairs(r=cutoff, output_type='ndarray')
    
    if len(pairs) == 0:
        return None

    data = np.ones(len(pairs))
    row = pairs[:, 0]
    col = pairs[:, 1]
    adj_matrix = csr_matrix((data, (row, col)), shape=(n_atoms, n_atoms))
    
    n_components, labels = connected_components(adj_matrix, directed=False)
    counts = np.bincount(labels)
    largest_label = np.argmax(counts)
    cluster_indices = np.where(labels == largest_label)[0]
    
    # --- B. 统计链的数量 (实时记录) ---
    chains_in_cluster = np.unique(atom_to_chain_map[cluster_indices])
    n_chains_in_cluster = len(chains_in_cluster)
    
    # --- C. 计算几何张量 ---
    if len(cluster_indices) < 10:
        return None
        
    cluster_pos = positions[cluster_indices]
    N_cluster = len(cluster_indices)
    
    r_cms = np.mean(cluster_pos, axis=0)
    dr = cluster_pos - r_cms
    C = np.dot(dr.T, dr) / N_cluster
    eigvals = np.linalg.eigvalsh(C)
    
    return {
        "frame": frame_idx,
        "eigvals": eigvals,
        "n_atoms": N_cluster,
        "n_chains": n_chains_in_cluster # 返回当前帧的链数
    }

# ==========================================================
# 2. 主分析函数
# ==========================================================

def run_analysis(tpr_path, xtc_path, selection="protein", cutoff=8.0, 
                 temp_K=310.0, block_size_frames=2000, start=0, end=None, step=1, n_jobs=-1):
    
    kB = 1.380649e-23
    u = mda.Universe(tpr_path, xtc_path)
    prot = u.select_atoms(selection)
    dt = u.trajectory.dt
    
    print(f"开始分析... 温度: {temp_K} K")
    print(f"Selection: {selection}")
    print(f"Block Size: {block_size_frames}, Cluster Cutoff: {cutoff} A")

    # 1. 建立链映射 (Topology Recognition)
    fragments = prot.fragments
    n_total_chains = len(fragments)
    atom_to_chain_map = np.zeros(prot.n_atoms, dtype=int)
    for i, frag in enumerate(fragments):
        _, _, mask = np.intersect1d(frag.indices, prot.indices, return_indices=True)
        atom_to_chain_map[mask] = i

    # 2. 准备并行任务
    trajectory = u.trajectory[start:end:step]
    n_frames = len(trajectory)
    task_args = []
    for ts in trajectory:
        task_args.append((
            ts.frame, 
            prot.positions.copy(), 
            atom_to_chain_map,
            cutoff, 
            temp_K, 
            kB
        ))
    
    # 3. 执行并行计算
    if n_jobs == -1: n_jobs = cpu_count()
    print("正在收集轨迹几何数据...")
    
    results_raw = []
    with Pool(processes=n_jobs) as pool:
        for i, res in enumerate(pool.imap(process_frame_task, task_args), 1):
            results_raw.append(res)
            if i % 10 == 0 or i == n_frames:
                sys.stdout.write(f"\r处理进度: {i}/{n_frames} ({i/n_frames*100:.1f}%)")
                sys.stdout.flush()
    
    print() 

    # 4. 数据整理
    valid_results = [r for r in results_raw if r is not None]
    if not valid_results:
        print("错误：未识别到有效团簇")
        return

    avg_atoms = np.mean([r['n_atoms'] for r in valid_results])
    avg_chains_total = np.mean([r['n_chains'] for r in valid_results])
    
    print(f"平均团簇包含粒子数: {avg_atoms:.1f} (总粒子数: {len(prot)})")
    print(f"平均团簇包含链数量: {avg_chains_total:.1f} (系统总链数: {n_total_chains})")
    print("数据收集完成。开始分块计算表面张力并分析链数变化...")
    
    # 5. 分块计算
    eigvals_hist = np.array([r['eigvals'] for r in valid_results])
    frames_hist = np.array([r['frame'] for r in valid_results])
    chains_hist = np.array([r['n_chains'] for r in valid_results])
    
    for i in range(0, len(valid_results), block_size_frames):
        block_eig = eigvals_hist[i : i + block_size_frames]
        block_frames = frames_hist[i : i + block_size_frames]
        block_chains = chains_hist[i : i + block_size_frames]
        
        if len(block_eig) < 100:
            continue
            
        # 半径 R
        raw_axes = np.sqrt(5 * block_eig)
        R_inst = np.prod(raw_axes, axis=1)**(1.0/3.0)
        R_avg = np.mean(R_inst)
        
        # 链数统计
        c_avg = np.mean(block_chains)
        c_min = np.min(block_chains)
        c_max = np.max(block_chains)
        
        # 表面张力计算
        l1, l2, l3 = block_eig[:, 0], block_eig[:, 1], block_eig[:, 2]
        a = R_avg * (l1**(1/3)) / ((l2 * l3)**(1/6))
        b = R_avg * (l2**(1/3)) / ((l1 * l3)**(1/6))
        c = R_avg * (l3**(1/3)) / ((l1 * l2)**(1/6))
        da, db, dc = a - R_avg, b - R_avg, c - R_avg
        var_plus = (np.mean((da + db)**2) + np.mean((db + dc)**2) + np.mean((dc + da)**2)) / 3.0
        var_minus = (np.mean((da - db)**2) + np.mean((db - dc)**2) + np.mean((dc - da)**2)) / 3.0
        
        ang2_to_m2 = 1e-20
        gamma_20 = (15 * kB * temp_K) / (16 * np.pi * ang2_to_m2 * var_plus) * 1000
        gamma_22 = (45 * kB * temp_K) / (16 * np.pi * ang2_to_m2 * var_minus) * 1000
        gamma_avg = (gamma_20 + gamma_22) / 2.0
        
        # 格式化输出
        start_ns = block_frames[0] * dt / 1000
        end_ns = block_frames[-1] * dt / 1000
        
        print(f"Block [{start_ns:.1f}-{end_ns:.1f} ns]: "
              f"Chains={c_avg:.1f}[{c_min}-{c_max}], R={R_avg:.1f}Å, "
              f"γ={gamma_avg:.3f} mN/m (γ20={gamma_20:.3f}, γ22={gamma_22:.3f})")

# ==========================================================
# 3. 运行设置
# ==========================================================

if __name__ == "__main__":
    # 请确保路径正确
    TPR = "/data/jychen/MD_projects/curvature_phase/Alpha-Synuclein/martini3001/Martini_alhx/mdrun/M3IDP/W_cluster/alhx_0_re1/TR_Mdvwhole/pbc.tpr"
    XTC = "/data/jychen/MD_projects/curvature_phase/Alpha-Synuclein/martini3001/Martini_alhx/mdrun/M3IDP/W_cluster/alhx_0_re1/TR_Mdvwhole/clus_centered.xtc"
    
    run_analysis(
        tpr_path=TPR,
        xtc_path=XTC,
        selection="protein",
        cutoff=8.0,
        temp_K=310.0,
        block_size_frames=2000, 
        start=10000,
        step=1,
        n_jobs=-1 
    )

开始分析... 温度: 310.0 K
Selection: protein
Block Size: 2000, Cluster Cutoff: 8.0 A
正在收集轨迹几何数据...
处理进度: 10001/10001 (100.0%)
平均团簇包含粒子数: 14685.4 (总粒子数: 14750)
平均团簇包含链数量: 49.8 (系统总链数: 50)
数据收集完成。开始分块计算表面张力并分析链数变化...
Block [10000.0-11999.0 ns]: Chains=49.0[48-50], R=95.8Å, γ=0.269 mN/m (γ20=0.263, γ22=0.276)
Block [12000.0-13999.0 ns]: Chains=49.9[49-50], R=97.2Å, γ=0.514 mN/m (γ20=0.507, γ22=0.521)
Block [14000.0-15999.0 ns]: Chains=50.0[50-50], R=96.7Å, γ=0.691 mN/m (γ20=0.684, γ22=0.699)
Block [16000.0-17999.0 ns]: Chains=50.0[50-50], R=97.9Å, γ=0.470 mN/m (γ20=0.462, γ22=0.477)
Block [18000.0-19999.0 ns]: Chains=50.0[49-50], R=97.6Å, γ=0.384 mN/m (γ20=0.377, γ22=0.391)


In [4]:
import MDAnalysis as mda
import numpy as np
import sys
from scipy.spatial import cKDTree
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from multiprocessing import Pool, cpu_count
import time

# ==========================================================
# 1. 核心工作函数 (单帧处理)
# ==========================================================

def process_frame_task(args):
    """
    处理单帧的并行函数
    """
    frame_idx, positions, atom_to_chain_map, cutoff, temp_K, kB = args
    
    n_atoms = len(positions)
    if n_atoms == 0:
        return None

    # --- A. 空间团簇识别 (KDTree) ---
    tree = cKDTree(positions)
    pairs = tree.query_pairs(r=cutoff, output_type='ndarray')
    
    if len(pairs) == 0:
        return None

    data = np.ones(len(pairs))
    row = pairs[:, 0]
    col = pairs[:, 1]
    adj_matrix = csr_matrix((data, (row, col)), shape=(n_atoms, n_atoms))
    
    n_components, labels = connected_components(adj_matrix, directed=False)
    counts = np.bincount(labels)
    largest_label = np.argmax(counts)
    cluster_indices = np.where(labels == largest_label)[0]
    
    # --- B. 统计链的数量 (实时记录) ---
    chains_in_cluster = np.unique(atom_to_chain_map[cluster_indices])
    n_chains_in_cluster = len(chains_in_cluster)
    
    # --- C. 计算几何张量 ---
    if len(cluster_indices) < 10:
        return None
        
    cluster_pos = positions[cluster_indices]
    N_cluster = len(cluster_indices)
    
    r_cms = np.mean(cluster_pos, axis=0)
    dr = cluster_pos - r_cms
    C = np.dot(dr.T, dr) / N_cluster
    eigvals = np.linalg.eigvalsh(C)
    
    return {
        "frame": frame_idx,
        "eigvals": eigvals,
        "n_atoms": N_cluster,
        "n_chains": n_chains_in_cluster # 返回当前帧的链数
    }

# ==========================================================
# 2. 主分析函数
# ==========================================================

def run_analysis(tpr_path, xtc_path, selection="protein", cutoff=8.0, 
                 temp_K=310.0, block_size_frames=2000, start=0, end=None, step=1, n_jobs=-1):
    
    kB = 1.380649e-23
    u = mda.Universe(tpr_path, xtc_path)
    prot = u.select_atoms(selection)
    dt = u.trajectory.dt
    
    print(f"开始分析... 温度: {temp_K} K")
    print(f"Selection: {selection}")
    print(f"Block Size: {block_size_frames}, Cluster Cutoff: {cutoff} A")

    # 1. 建立链映射 (Topology Recognition)
    fragments = prot.fragments
    n_total_chains = len(fragments)
    atom_to_chain_map = np.zeros(prot.n_atoms, dtype=int)
    for i, frag in enumerate(fragments):
        _, _, mask = np.intersect1d(frag.indices, prot.indices, return_indices=True)
        atom_to_chain_map[mask] = i

    # 2. 准备并行任务
    trajectory = u.trajectory[start:end:step]
    n_frames = len(trajectory)
    task_args = []
    for ts in trajectory:
        task_args.append((
            ts.frame, 
            prot.positions.copy(), 
            atom_to_chain_map,
            cutoff, 
            temp_K, 
            kB
        ))
    
    # 3. 执行并行计算
    if n_jobs == -1: n_jobs = cpu_count()
    print("正在收集轨迹几何数据...")
    
    results_raw = []
    with Pool(processes=n_jobs) as pool:
        for i, res in enumerate(pool.imap(process_frame_task, task_args), 1):
            results_raw.append(res)
            if i % 10 == 0 or i == n_frames:
                sys.stdout.write(f"\r处理进度: {i}/{n_frames} ({i/n_frames*100:.1f}%)")
                sys.stdout.flush()
    
    print() 

    # 4. 数据整理
    valid_results = [r for r in results_raw if r is not None]
    if not valid_results:
        print("错误：未识别到有效团簇")
        return

    avg_atoms = np.mean([r['n_atoms'] for r in valid_results])
    avg_chains_total = np.mean([r['n_chains'] for r in valid_results])
    
    print(f"平均团簇包含粒子数: {avg_atoms:.1f} (总粒子数: {len(prot)})")
    print(f"平均团簇包含链数量: {avg_chains_total:.1f} (系统总链数: {n_total_chains})")
    print("数据收集完成。开始分块计算表面张力并分析链数变化...")
    
    # 5. 分块计算
    eigvals_hist = np.array([r['eigvals'] for r in valid_results])
    frames_hist = np.array([r['frame'] for r in valid_results])
    chains_hist = np.array([r['n_chains'] for r in valid_results])
    
    for i in range(0, len(valid_results), block_size_frames):
        block_eig = eigvals_hist[i : i + block_size_frames]
        block_frames = frames_hist[i : i + block_size_frames]
        block_chains = chains_hist[i : i + block_size_frames]
        
        if len(block_eig) < 100:
            continue
            
        # 半径 R
        raw_axes = np.sqrt(5 * block_eig)
        R_inst = np.prod(raw_axes, axis=1)**(1.0/3.0)
        R_avg = np.mean(R_inst)
        
        # 链数统计
        c_avg = np.mean(block_chains)
        c_min = np.min(block_chains)
        c_max = np.max(block_chains)
        
        # 表面张力计算
        l1, l2, l3 = block_eig[:, 0], block_eig[:, 1], block_eig[:, 2]
        a = R_avg * (l1**(1/3)) / ((l2 * l3)**(1/6))
        b = R_avg * (l2**(1/3)) / ((l1 * l3)**(1/6))
        c = R_avg * (l3**(1/3)) / ((l1 * l2)**(1/6))
        da, db, dc = a - R_avg, b - R_avg, c - R_avg
        var_plus = (np.mean((da + db)**2) + np.mean((db + dc)**2) + np.mean((dc + da)**2)) / 3.0
        var_minus = (np.mean((da - db)**2) + np.mean((db - dc)**2) + np.mean((dc - da)**2)) / 3.0
        
        ang2_to_m2 = 1e-20
        gamma_20 = (15 * kB * temp_K) / (16 * np.pi * ang2_to_m2 * var_plus) * 1000
        gamma_22 = (45 * kB * temp_K) / (16 * np.pi * ang2_to_m2 * var_minus) * 1000
        gamma_avg = (gamma_20 + gamma_22) / 2.0
        
        # 格式化输出
        start_ns = block_frames[0] * dt / 1000
        end_ns = block_frames[-1] * dt / 1000
        
        print(f"Block [{start_ns:.1f}-{end_ns:.1f} ns]: "
              f"Chains={c_avg:.1f}[{c_min}-{c_max}], R={R_avg:.1f}Å, "
              f"γ={gamma_avg:.3f} mN/m (γ20={gamma_20:.3f}, γ22={gamma_22:.3f})")

# ==========================================================
# 3. 运行设置
# ==========================================================

if __name__ == "__main__":
    # 请确保路径正确
    TPR = "/data/jychen/MD_projects/curvature_phase/Alpha-Synuclein/martini3001/Martini_alhx/mdrun/M3IDP/W_cluster/alhx_0_re2/TR_Mdvwhole/pbc.tpr"
    XTC = "/data/jychen/MD_projects/curvature_phase/Alpha-Synuclein/martini3001/Martini_alhx/mdrun/M3IDP/W_cluster/alhx_0_re2/TR_Mdvwhole/clus_centered.xtc"
    
    run_analysis(
        tpr_path=TPR,
        xtc_path=XTC,
        selection="protein",
        cutoff=8.0,
        temp_K=310.0,
        block_size_frames=2000, 
        start=10000,
        step=1,
        n_jobs=-1 
    )

开始分析... 温度: 310.0 K
Selection: protein
Block Size: 2000, Cluster Cutoff: 8.0 A
正在收集轨迹几何数据...
处理进度: 10001/10001 (100.0%)
平均团簇包含粒子数: 14638.3 (总粒子数: 14750)
平均团簇包含链数量: 49.6 (系统总链数: 50)
数据收集完成。开始分块计算表面张力并分析链数变化...
Block [10000.0-11999.0 ns]: Chains=49.0[49-50], R=96.8Å, γ=0.739 mN/m (γ20=0.731, γ22=0.747)
Block [12000.0-13999.0 ns]: Chains=49.8[49-50], R=96.6Å, γ=1.348 mN/m (γ20=1.339, γ22=1.357)
Block [14000.0-15999.0 ns]: Chains=50.0[50-50], R=97.5Å, γ=0.659 mN/m (γ20=0.651, γ22=0.667)
Block [16000.0-17999.0 ns]: Chains=50.0[50-50], R=96.9Å, γ=0.511 mN/m (γ20=0.503, γ22=0.519)
Block [18000.0-19999.0 ns]: Chains=49.3[48-50], R=96.8Å, γ=0.449 mN/m (γ20=0.439, γ22=0.459)


In [5]:
import MDAnalysis as mda
import numpy as np
import sys
from scipy.spatial import cKDTree
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from multiprocessing import Pool, cpu_count
import time

# ==========================================================
# 1. 核心工作函数 (单帧处理)
# ==========================================================

def process_frame_task(args):
    """
    处理单帧的并行函数
    """
    frame_idx, positions, atom_to_chain_map, cutoff, temp_K, kB = args
    
    n_atoms = len(positions)
    if n_atoms == 0:
        return None

    # --- A. 空间团簇识别 (KDTree) ---
    tree = cKDTree(positions)
    pairs = tree.query_pairs(r=cutoff, output_type='ndarray')
    
    if len(pairs) == 0:
        return None

    data = np.ones(len(pairs))
    row = pairs[:, 0]
    col = pairs[:, 1]
    adj_matrix = csr_matrix((data, (row, col)), shape=(n_atoms, n_atoms))
    
    n_components, labels = connected_components(adj_matrix, directed=False)
    counts = np.bincount(labels)
    largest_label = np.argmax(counts)
    cluster_indices = np.where(labels == largest_label)[0]
    
    # --- B. 统计链的数量 (实时记录) ---
    chains_in_cluster = np.unique(atom_to_chain_map[cluster_indices])
    n_chains_in_cluster = len(chains_in_cluster)
    
    # --- C. 计算几何张量 ---
    if len(cluster_indices) < 10:
        return None
        
    cluster_pos = positions[cluster_indices]
    N_cluster = len(cluster_indices)
    
    r_cms = np.mean(cluster_pos, axis=0)
    dr = cluster_pos - r_cms
    C = np.dot(dr.T, dr) / N_cluster
    eigvals = np.linalg.eigvalsh(C)
    
    return {
        "frame": frame_idx,
        "eigvals": eigvals,
        "n_atoms": N_cluster,
        "n_chains": n_chains_in_cluster # 返回当前帧的链数
    }

# ==========================================================
# 2. 主分析函数
# ==========================================================

def run_analysis(tpr_path, xtc_path, selection="protein", cutoff=8.0, 
                 temp_K=310.0, block_size_frames=2000, start=0, end=None, step=1, n_jobs=-1):
    
    kB = 1.380649e-23
    u = mda.Universe(tpr_path, xtc_path)
    prot = u.select_atoms(selection)
    dt = u.trajectory.dt
    
    print(f"开始分析... 温度: {temp_K} K")
    print(f"Selection: {selection}")
    print(f"Block Size: {block_size_frames}, Cluster Cutoff: {cutoff} A")

    # 1. 建立链映射 (Topology Recognition)
    fragments = prot.fragments
    n_total_chains = len(fragments)
    atom_to_chain_map = np.zeros(prot.n_atoms, dtype=int)
    for i, frag in enumerate(fragments):
        _, _, mask = np.intersect1d(frag.indices, prot.indices, return_indices=True)
        atom_to_chain_map[mask] = i

    # 2. 准备并行任务
    trajectory = u.trajectory[start:end:step]
    n_frames = len(trajectory)
    task_args = []
    for ts in trajectory:
        task_args.append((
            ts.frame, 
            prot.positions.copy(), 
            atom_to_chain_map,
            cutoff, 
            temp_K, 
            kB
        ))
    
    # 3. 执行并行计算
    if n_jobs == -1: n_jobs = cpu_count()
    print("正在收集轨迹几何数据...")
    
    results_raw = []
    with Pool(processes=n_jobs) as pool:
        for i, res in enumerate(pool.imap(process_frame_task, task_args), 1):
            results_raw.append(res)
            if i % 10 == 0 or i == n_frames:
                sys.stdout.write(f"\r处理进度: {i}/{n_frames} ({i/n_frames*100:.1f}%)")
                sys.stdout.flush()
    
    print() 

    # 4. 数据整理
    valid_results = [r for r in results_raw if r is not None]
    if not valid_results:
        print("错误：未识别到有效团簇")
        return

    avg_atoms = np.mean([r['n_atoms'] for r in valid_results])
    avg_chains_total = np.mean([r['n_chains'] for r in valid_results])
    
    print(f"平均团簇包含粒子数: {avg_atoms:.1f} (总粒子数: {len(prot)})")
    print(f"平均团簇包含链数量: {avg_chains_total:.1f} (系统总链数: {n_total_chains})")
    print("数据收集完成。开始分块计算表面张力并分析链数变化...")
    
    # 5. 分块计算
    eigvals_hist = np.array([r['eigvals'] for r in valid_results])
    frames_hist = np.array([r['frame'] for r in valid_results])
    chains_hist = np.array([r['n_chains'] for r in valid_results])
    
    for i in range(0, len(valid_results), block_size_frames):
        block_eig = eigvals_hist[i : i + block_size_frames]
        block_frames = frames_hist[i : i + block_size_frames]
        block_chains = chains_hist[i : i + block_size_frames]
        
        if len(block_eig) < 100:
            continue
            
        # 半径 R
        raw_axes = np.sqrt(5 * block_eig)
        R_inst = np.prod(raw_axes, axis=1)**(1.0/3.0)
        R_avg = np.mean(R_inst)
        
        # 链数统计
        c_avg = np.mean(block_chains)
        c_min = np.min(block_chains)
        c_max = np.max(block_chains)
        
        # 表面张力计算
        l1, l2, l3 = block_eig[:, 0], block_eig[:, 1], block_eig[:, 2]
        a = R_avg * (l1**(1/3)) / ((l2 * l3)**(1/6))
        b = R_avg * (l2**(1/3)) / ((l1 * l3)**(1/6))
        c = R_avg * (l3**(1/3)) / ((l1 * l2)**(1/6))
        da, db, dc = a - R_avg, b - R_avg, c - R_avg
        var_plus = (np.mean((da + db)**2) + np.mean((db + dc)**2) + np.mean((dc + da)**2)) / 3.0
        var_minus = (np.mean((da - db)**2) + np.mean((db - dc)**2) + np.mean((dc - da)**2)) / 3.0
        
        ang2_to_m2 = 1e-20
        gamma_20 = (15 * kB * temp_K) / (16 * np.pi * ang2_to_m2 * var_plus) * 1000
        gamma_22 = (45 * kB * temp_K) / (16 * np.pi * ang2_to_m2 * var_minus) * 1000
        gamma_avg = (gamma_20 + gamma_22) / 2.0
        
        # 格式化输出
        start_ns = block_frames[0] * dt / 1000
        end_ns = block_frames[-1] * dt / 1000
        
        print(f"Block [{start_ns:.1f}-{end_ns:.1f} ns]: "
              f"Chains={c_avg:.1f}[{c_min}-{c_max}], R={R_avg:.1f}Å, "
              f"γ={gamma_avg:.3f} mN/m (γ20={gamma_20:.3f}, γ22={gamma_22:.3f})")

# ==========================================================
# 3. 运行设置
# ==========================================================

if __name__ == "__main__":
    # 请确保路径正确
    TPR = "/data/jychen/MD_projects/curvature_phase/Alpha-Synuclein/martini3001/Martini_alhx/mdrun/M3IDP/W_cluster/alhx_700/TR_Mdvwhole/pbc.tpr"
    XTC = "/data/jychen/MD_projects/curvature_phase/Alpha-Synuclein/martini3001/Martini_alhx/mdrun/M3IDP/W_cluster/alhx_700/TR_Mdvwhole/clus_centered.xtc"
    
    run_analysis(
        tpr_path=TPR,
        xtc_path=XTC,
        selection="protein",
        cutoff=8.0,
        temp_K=310.0,
        block_size_frames=2000, 
        start=10000,
        step=1,
        n_jobs=-1 
    )

开始分析... 温度: 310.0 K
Selection: protein
Block Size: 2000, Cluster Cutoff: 8.0 A
正在收集轨迹几何数据...
处理进度: 10001/10001 (100.0%)
平均团簇包含粒子数: 14594.1 (总粒子数: 14750)
平均团簇包含链数量: 49.5 (系统总链数: 50)
数据收集完成。开始分块计算表面张力并分析链数变化...
Block [10000.0-11999.0 ns]: Chains=49.0[49-50], R=91.1Å, γ=0.212 mN/m (γ20=0.206, γ22=0.218)
Block [12000.0-13999.0 ns]: Chains=49.0[49-49], R=91.1Å, γ=0.233 mN/m (γ20=0.227, γ22=0.239)
Block [14000.0-15999.0 ns]: Chains=49.4[48-50], R=91.9Å, γ=0.298 mN/m (γ20=0.292, γ22=0.304)
Block [16000.0-17999.0 ns]: Chains=50.0[50-50], R=91.2Å, γ=0.389 mN/m (γ20=0.383, γ22=0.396)
Block [18000.0-19999.0 ns]: Chains=50.0[49-50], R=94.8Å, γ=0.621 mN/m (γ20=0.614, γ22=0.627)


In [6]:
import MDAnalysis as mda
import numpy as np
import sys
from scipy.spatial import cKDTree
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from multiprocessing import Pool, cpu_count
import time

# ==========================================================
# 1. 核心工作函数 (单帧处理)
# ==========================================================

def process_frame_task(args):
    """
    处理单帧的并行函数
    """
    frame_idx, positions, atom_to_chain_map, cutoff, temp_K, kB = args
    
    n_atoms = len(positions)
    if n_atoms == 0:
        return None

    # --- A. 空间团簇识别 (KDTree) ---
    tree = cKDTree(positions)
    pairs = tree.query_pairs(r=cutoff, output_type='ndarray')
    
    if len(pairs) == 0:
        return None

    data = np.ones(len(pairs))
    row = pairs[:, 0]
    col = pairs[:, 1]
    adj_matrix = csr_matrix((data, (row, col)), shape=(n_atoms, n_atoms))
    
    n_components, labels = connected_components(adj_matrix, directed=False)
    counts = np.bincount(labels)
    largest_label = np.argmax(counts)
    cluster_indices = np.where(labels == largest_label)[0]
    
    # --- B. 统计链的数量 (实时记录) ---
    chains_in_cluster = np.unique(atom_to_chain_map[cluster_indices])
    n_chains_in_cluster = len(chains_in_cluster)
    
    # --- C. 计算几何张量 ---
    if len(cluster_indices) < 10:
        return None
        
    cluster_pos = positions[cluster_indices]
    N_cluster = len(cluster_indices)
    
    r_cms = np.mean(cluster_pos, axis=0)
    dr = cluster_pos - r_cms
    C = np.dot(dr.T, dr) / N_cluster
    eigvals = np.linalg.eigvalsh(C)
    
    return {
        "frame": frame_idx,
        "eigvals": eigvals,
        "n_atoms": N_cluster,
        "n_chains": n_chains_in_cluster # 返回当前帧的链数
    }

# ==========================================================
# 2. 主分析函数
# ==========================================================

def run_analysis(tpr_path, xtc_path, selection="protein", cutoff=8.0, 
                 temp_K=310.0, block_size_frames=2000, start=0, end=None, step=1, n_jobs=-1):
    
    kB = 1.380649e-23
    u = mda.Universe(tpr_path, xtc_path)
    prot = u.select_atoms(selection)
    dt = u.trajectory.dt
    
    print(f"开始分析... 温度: {temp_K} K")
    print(f"Selection: {selection}")
    print(f"Block Size: {block_size_frames}, Cluster Cutoff: {cutoff} A")

    # 1. 建立链映射 (Topology Recognition)
    fragments = prot.fragments
    n_total_chains = len(fragments)
    atom_to_chain_map = np.zeros(prot.n_atoms, dtype=int)
    for i, frag in enumerate(fragments):
        _, _, mask = np.intersect1d(frag.indices, prot.indices, return_indices=True)
        atom_to_chain_map[mask] = i

    # 2. 准备并行任务
    trajectory = u.trajectory[start:end:step]
    n_frames = len(trajectory)
    task_args = []
    for ts in trajectory:
        task_args.append((
            ts.frame, 
            prot.positions.copy(), 
            atom_to_chain_map,
            cutoff, 
            temp_K, 
            kB
        ))
    
    # 3. 执行并行计算
    if n_jobs == -1: n_jobs = cpu_count()
    print("正在收集轨迹几何数据...")
    
    results_raw = []
    with Pool(processes=n_jobs) as pool:
        for i, res in enumerate(pool.imap(process_frame_task, task_args), 1):
            results_raw.append(res)
            if i % 10 == 0 or i == n_frames:
                sys.stdout.write(f"\r处理进度: {i}/{n_frames} ({i/n_frames*100:.1f}%)")
                sys.stdout.flush()
    
    print() 

    # 4. 数据整理
    valid_results = [r for r in results_raw if r is not None]
    if not valid_results:
        print("错误：未识别到有效团簇")
        return

    avg_atoms = np.mean([r['n_atoms'] for r in valid_results])
    avg_chains_total = np.mean([r['n_chains'] for r in valid_results])
    
    print(f"平均团簇包含粒子数: {avg_atoms:.1f} (总粒子数: {len(prot)})")
    print(f"平均团簇包含链数量: {avg_chains_total:.1f} (系统总链数: {n_total_chains})")
    print("数据收集完成。开始分块计算表面张力并分析链数变化...")
    
    # 5. 分块计算
    eigvals_hist = np.array([r['eigvals'] for r in valid_results])
    frames_hist = np.array([r['frame'] for r in valid_results])
    chains_hist = np.array([r['n_chains'] for r in valid_results])
    
    for i in range(0, len(valid_results), block_size_frames):
        block_eig = eigvals_hist[i : i + block_size_frames]
        block_frames = frames_hist[i : i + block_size_frames]
        block_chains = chains_hist[i : i + block_size_frames]
        
        if len(block_eig) < 100:
            continue
            
        # 半径 R
        raw_axes = np.sqrt(5 * block_eig)
        R_inst = np.prod(raw_axes, axis=1)**(1.0/3.0)
        R_avg = np.mean(R_inst)
        
        # 链数统计
        c_avg = np.mean(block_chains)
        c_min = np.min(block_chains)
        c_max = np.max(block_chains)
        
        # 表面张力计算
        l1, l2, l3 = block_eig[:, 0], block_eig[:, 1], block_eig[:, 2]
        a = R_avg * (l1**(1/3)) / ((l2 * l3)**(1/6))
        b = R_avg * (l2**(1/3)) / ((l1 * l3)**(1/6))
        c = R_avg * (l3**(1/3)) / ((l1 * l2)**(1/6))
        da, db, dc = a - R_avg, b - R_avg, c - R_avg
        var_plus = (np.mean((da + db)**2) + np.mean((db + dc)**2) + np.mean((dc + da)**2)) / 3.0
        var_minus = (np.mean((da - db)**2) + np.mean((db - dc)**2) + np.mean((dc - da)**2)) / 3.0
        
        ang2_to_m2 = 1e-20
        gamma_20 = (15 * kB * temp_K) / (16 * np.pi * ang2_to_m2 * var_plus) * 1000
        gamma_22 = (45 * kB * temp_K) / (16 * np.pi * ang2_to_m2 * var_minus) * 1000
        gamma_avg = (gamma_20 + gamma_22) / 2.0
        
        # 格式化输出
        start_ns = block_frames[0] * dt / 1000
        end_ns = block_frames[-1] * dt / 1000
        
        print(f"Block [{start_ns:.1f}-{end_ns:.1f} ns]: "
              f"Chains={c_avg:.1f}[{c_min}-{c_max}], R={R_avg:.1f}Å, "
              f"γ={gamma_avg:.3f} mN/m (γ20={gamma_20:.3f}, γ22={gamma_22:.3f})")

# ==========================================================
# 3. 运行设置
# ==========================================================

if __name__ == "__main__":
    # 请确保路径正确
    TPR = "/data/jychen/MD_projects/curvature_phase/Alpha-Synuclein/martini3001/Martini_alhx/mdrun/M3IDP/W_cluster/alhx_700_re1/TR_Mdvwhole/pbc.tpr"
    XTC = "/data/jychen/MD_projects/curvature_phase/Alpha-Synuclein/martini3001/Martini_alhx/mdrun/M3IDP/W_cluster/alhx_700_re1/TR_Mdvwhole/clus_centered.xtc"
    
    run_analysis(
        tpr_path=TPR,
        xtc_path=XTC,
        selection="protein",
        cutoff=8.0,
        temp_K=310.0,
        block_size_frames=2000, 
        start=10000,
        step=1,
        n_jobs=-1 
    )

开始分析... 温度: 310.0 K
Selection: protein
Block Size: 2000, Cluster Cutoff: 8.0 A
正在收集轨迹几何数据...
处理进度: 10001/10001 (100.0%)
平均团簇包含粒子数: 14622.5 (总粒子数: 14750)
平均团簇包含链数量: 49.6 (系统总链数: 50)
数据收集完成。开始分块计算表面张力并分析链数变化...
Block [10000.0-11999.0 ns]: Chains=49.0[48-50], R=93.3Å, γ=0.141 mN/m (γ20=0.136, γ22=0.146)
Block [12000.0-13999.0 ns]: Chains=49.0[49-49], R=91.5Å, γ=0.234 mN/m (γ20=0.227, γ22=0.240)
Block [14000.0-15999.0 ns]: Chains=49.9[49-50], R=92.7Å, γ=0.211 mN/m (γ20=0.205, γ22=0.217)
Block [16000.0-17999.0 ns]: Chains=50.0[50-50], R=90.7Å, γ=0.276 mN/m (γ20=0.269, γ22=0.282)
Block [18000.0-19999.0 ns]: Chains=50.0[50-50], R=90.2Å, γ=0.275 mN/m (γ20=0.268, γ22=0.281)


In [7]:
import MDAnalysis as mda
import numpy as np
import sys
from scipy.spatial import cKDTree
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from multiprocessing import Pool, cpu_count
import time

# ==========================================================
# 1. 核心工作函数 (单帧处理)
# ==========================================================

def process_frame_task(args):
    """
    处理单帧的并行函数
    """
    frame_idx, positions, atom_to_chain_map, cutoff, temp_K, kB = args
    
    n_atoms = len(positions)
    if n_atoms == 0:
        return None

    # --- A. 空间团簇识别 (KDTree) ---
    tree = cKDTree(positions)
    pairs = tree.query_pairs(r=cutoff, output_type='ndarray')
    
    if len(pairs) == 0:
        return None

    data = np.ones(len(pairs))
    row = pairs[:, 0]
    col = pairs[:, 1]
    adj_matrix = csr_matrix((data, (row, col)), shape=(n_atoms, n_atoms))
    
    n_components, labels = connected_components(adj_matrix, directed=False)
    counts = np.bincount(labels)
    largest_label = np.argmax(counts)
    cluster_indices = np.where(labels == largest_label)[0]
    
    # --- B. 统计链的数量 (实时记录) ---
    chains_in_cluster = np.unique(atom_to_chain_map[cluster_indices])
    n_chains_in_cluster = len(chains_in_cluster)
    
    # --- C. 计算几何张量 ---
    if len(cluster_indices) < 10:
        return None
        
    cluster_pos = positions[cluster_indices]
    N_cluster = len(cluster_indices)
    
    r_cms = np.mean(cluster_pos, axis=0)
    dr = cluster_pos - r_cms
    C = np.dot(dr.T, dr) / N_cluster
    eigvals = np.linalg.eigvalsh(C)
    
    return {
        "frame": frame_idx,
        "eigvals": eigvals,
        "n_atoms": N_cluster,
        "n_chains": n_chains_in_cluster # 返回当前帧的链数
    }

# ==========================================================
# 2. 主分析函数
# ==========================================================

def run_analysis(tpr_path, xtc_path, selection="protein", cutoff=8.0, 
                 temp_K=310.0, block_size_frames=2000, start=0, end=None, step=1, n_jobs=-1):
    
    kB = 1.380649e-23
    u = mda.Universe(tpr_path, xtc_path)
    prot = u.select_atoms(selection)
    dt = u.trajectory.dt
    
    print(f"开始分析... 温度: {temp_K} K")
    print(f"Selection: {selection}")
    print(f"Block Size: {block_size_frames}, Cluster Cutoff: {cutoff} A")

    # 1. 建立链映射 (Topology Recognition)
    fragments = prot.fragments
    n_total_chains = len(fragments)
    atom_to_chain_map = np.zeros(prot.n_atoms, dtype=int)
    for i, frag in enumerate(fragments):
        _, _, mask = np.intersect1d(frag.indices, prot.indices, return_indices=True)
        atom_to_chain_map[mask] = i

    # 2. 准备并行任务
    trajectory = u.trajectory[start:end:step]
    n_frames = len(trajectory)
    task_args = []
    for ts in trajectory:
        task_args.append((
            ts.frame, 
            prot.positions.copy(), 
            atom_to_chain_map,
            cutoff, 
            temp_K, 
            kB
        ))
    
    # 3. 执行并行计算
    if n_jobs == -1: n_jobs = cpu_count()
    print("正在收集轨迹几何数据...")
    
    results_raw = []
    with Pool(processes=n_jobs) as pool:
        for i, res in enumerate(pool.imap(process_frame_task, task_args), 1):
            results_raw.append(res)
            if i % 10 == 0 or i == n_frames:
                sys.stdout.write(f"\r处理进度: {i}/{n_frames} ({i/n_frames*100:.1f}%)")
                sys.stdout.flush()
    
    print() 

    # 4. 数据整理
    valid_results = [r for r in results_raw if r is not None]
    if not valid_results:
        print("错误：未识别到有效团簇")
        return

    avg_atoms = np.mean([r['n_atoms'] for r in valid_results])
    avg_chains_total = np.mean([r['n_chains'] for r in valid_results])
    
    print(f"平均团簇包含粒子数: {avg_atoms:.1f} (总粒子数: {len(prot)})")
    print(f"平均团簇包含链数量: {avg_chains_total:.1f} (系统总链数: {n_total_chains})")
    print("数据收集完成。开始分块计算表面张力并分析链数变化...")
    
    # 5. 分块计算
    eigvals_hist = np.array([r['eigvals'] for r in valid_results])
    frames_hist = np.array([r['frame'] for r in valid_results])
    chains_hist = np.array([r['n_chains'] for r in valid_results])
    
    for i in range(0, len(valid_results), block_size_frames):
        block_eig = eigvals_hist[i : i + block_size_frames]
        block_frames = frames_hist[i : i + block_size_frames]
        block_chains = chains_hist[i : i + block_size_frames]
        
        if len(block_eig) < 100:
            continue
            
        # 半径 R
        raw_axes = np.sqrt(5 * block_eig)
        R_inst = np.prod(raw_axes, axis=1)**(1.0/3.0)
        R_avg = np.mean(R_inst)
        
        # 链数统计
        c_avg = np.mean(block_chains)
        c_min = np.min(block_chains)
        c_max = np.max(block_chains)
        
        # 表面张力计算
        l1, l2, l3 = block_eig[:, 0], block_eig[:, 1], block_eig[:, 2]
        a = R_avg * (l1**(1/3)) / ((l2 * l3)**(1/6))
        b = R_avg * (l2**(1/3)) / ((l1 * l3)**(1/6))
        c = R_avg * (l3**(1/3)) / ((l1 * l2)**(1/6))
        da, db, dc = a - R_avg, b - R_avg, c - R_avg
        var_plus = (np.mean((da + db)**2) + np.mean((db + dc)**2) + np.mean((dc + da)**2)) / 3.0
        var_minus = (np.mean((da - db)**2) + np.mean((db - dc)**2) + np.mean((dc - da)**2)) / 3.0
        
        ang2_to_m2 = 1e-20
        gamma_20 = (15 * kB * temp_K) / (16 * np.pi * ang2_to_m2 * var_plus) * 1000
        gamma_22 = (45 * kB * temp_K) / (16 * np.pi * ang2_to_m2 * var_minus) * 1000
        gamma_avg = (gamma_20 + gamma_22) / 2.0
        
        # 格式化输出
        start_ns = block_frames[0] * dt / 1000
        end_ns = block_frames[-1] * dt / 1000
        
        print(f"Block [{start_ns:.1f}-{end_ns:.1f} ns]: "
              f"Chains={c_avg:.1f}[{c_min}-{c_max}], R={R_avg:.1f}Å, "
              f"γ={gamma_avg:.3f} mN/m (γ20={gamma_20:.3f}, γ22={gamma_22:.3f})")

# ==========================================================
# 3. 运行设置
# ==========================================================

if __name__ == "__main__":
    # 请确保路径正确
    TPR = "/data/jychen/MD_projects/curvature_phase/Alpha-Synuclein/martini3001/Martini_alhx/mdrun/M3IDP/W_cluster/alhx_700_re2/TR_Mdvwhole/pbc.tpr"
    XTC = "/data/jychen/MD_projects/curvature_phase/Alpha-Synuclein/martini3001/Martini_alhx/mdrun/M3IDP/W_cluster/alhx_700_re2/TR_Mdvwhole/clus_centered.xtc"
    
    run_analysis(
        tpr_path=TPR,
        xtc_path=XTC,
        selection="protein",
        cutoff=8.0,
        temp_K=310.0,
        block_size_frames=2000, 
        start=10000,
        step=1,
        n_jobs=-1 
    )

开始分析... 温度: 310.0 K
Selection: protein
Block Size: 2000, Cluster Cutoff: 8.0 A
正在收集轨迹几何数据...
处理进度: 10001/10001 (100.0%)
平均团簇包含粒子数: 14365.7 (总粒子数: 14750)
平均团簇包含链数量: 48.7 (系统总链数: 50)
数据收集完成。开始分块计算表面张力并分析链数变化...
Block [10000.0-11999.0 ns]: Chains=47.9[47-48], R=95.0Å, γ=0.198 mN/m (γ20=0.192, γ22=0.203)
Block [12000.0-13999.0 ns]: Chains=48.1[48-49], R=91.3Å, γ=0.192 mN/m (γ20=0.186, γ22=0.198)
Block [14000.0-15999.0 ns]: Chains=48.5[48-50], R=93.4Å, γ=0.193 mN/m (γ20=0.187, γ22=0.199)
Block [16000.0-17999.0 ns]: Chains=49.1[49-50], R=94.0Å, γ=0.332 mN/m (γ20=0.325, γ22=0.338)
Block [18000.0-19999.0 ns]: Chains=49.9[49-50], R=92.6Å, γ=0.499 mN/m (γ20=0.492, γ22=0.506)
